# Training model with PneumoniaMNIST Dataset

In [17]:
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
from pathlib import Path

import medmnist
from medmnist import INFO, Evaluator

### Credit

In [18]:
print(f"MedMNIST v{medmnist.__version__} @ {medmnist.HOMEPAGE}")

MedMNIST v3.0.2 @ https://github.com/MedMNIST/MedMNIST/


### Loading Dataset

In [19]:
data_flag = 'pneumoniamnist' # Dataset being used. This is a 2D dataset using 28x28 images.
download = True # downloads the PneumoniaMNIST dataset if it isn't downloaded

NUM_EPOCHS = 3 # Sets the number of training passes though the entire training dataset
BATCH_SIZE = 128 # Processes 128 images at a time
lr = 0.001 # Learning rate

# Path for the dataset
DATA_ROOT = Path("pneumoniaMNIST_data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

info = INFO[data_flag]
task = info['task'] # Checks the type of ML task
n_channels = info['n_channels'] # Finds the color scale of the dataset: 1 for grayscale, 3 for RGB
n_classes = len(info['label']) # Gets the number of labels/classes in the dataset

DataClass = getattr(medmnist, info['python_class']) # Retrieves the dataset. Since info is set to INFO[data_flag], python_class is PneumoniaMNIST.


In [20]:
# preprocessing
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

"""
Transforms images to tensors then normalizes their pixel values between -1 and 1.
ToTensor takes the pixel value and divides by 255
Normalize takes that value, subtracts .5, then divides by .5
"""

# load the data
train_dataset = DataClass(split='train', transform=data_transform, download=download, root=str(DATA_ROOT))
test_dataset = DataClass(split='test', transform=data_transform, download=download, root=str(DATA_ROOT))

# Creates copy of training dataset without transforming
pil_dataset = DataClass(split='train', download=download, root=str(DATA_ROOT)) 

# encapsulate data into dataloader form

# Shuffles batches for each epoche for training
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Non-shuffled training batches for measuring train accuracy
train_loader_at_eval = data.DataLoader(dataset=train_dataset, batch_size=2*BATCH_SIZE, shuffle=False)

# Non-shuffled for measuring test accuracy
test_loader = data.DataLoader(dataset=test_dataset, batch_size=2*BATCH_SIZE, shuffle=False)

### Print Training and Test Datasets

In [21]:
print(train_dataset)
print("="*55)
print(test_dataset)

Dataset PneumoniaMNIST of size 28 (pneumoniamnist)
    Number of datapoints: 4708
    Root location: pneumoniaMNIST_data
    Split: train
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'normal', '1': 'pneumonia'}
    Number of samples: {'train': 4708, 'val': 524, 'test': 624}
    Description: The PneumoniaMNIST is based on a prior dataset of 5,856 pediatric chest X-Ray images. The task is binary-class classification of pneumonia against normal. We split the source training set with a ratio of 9:1 into training and validation set and use its source validation set as the test set. The source images are gray-scale, and their sizes are (384−2,916)×(127−2,713). We center-crop the images and resize them into 1×28×28.
    License: CC BY 4.0
Dataset PneumoniaMNIST of size 28 (pneumoniamnist)
    Number of datapoints: 624
    Root location: pneumoniaMNIST_data
    Split: test
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'normal', '

### CNN Model

In [22]:
class Net(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(Net, self).__init__()

        # Extracts simple patterns
        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3), 
            # Takes in number of channels, learns 16 filters, 
            # and each filter looks at a 3x3 region at a time

            nn.BatchNorm2d(16),
            # Normalizes the 16 output feature maps during training for stability
            
            nn.ReLU())
            # Adds nonlinearity. Allows NN to learn complicated patterns as
            # it prevents every layer from becoming a linear transformation

        # Extracts more patterns, then shrinks image
        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
            # Shrinks image width and height by about half
            # Keeps the strongest value from each 2x2 region

        # Learns more complex patterns
        self.layer3 = nn.Sequential(
            nn.Conv2d(16, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU())

        # Refines those patterns
        self.layer4 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU())

        # Keeps size then shrinks again
        self.layer5 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            # padding adds a one-pixel border around input
            # before applying filter, keeping image size the same
    
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))

        # Makes final class prediction
        self.fc = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            # With each image having shape 64x4x4 (1024 values),
            # the values are converted into 128 learned features.

            nn.Linear(128, num_classes))
            # Creates final output layer i.e. the two raw scores.

    # Defines how the image moves through the model.
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)

        x = x.view(x.size(0), -1)
        # Flattens each image into a list of 1024 values

        x = self.fc(x)
        return x

model = Net(in_channels=n_channels, num_classes=n_classes)
    
# define loss function and optimizer
if task == "multi-label, binary-class":
    criterion = nn.BCEWithLogitsLoss()
    # Treats each label as a separate y/n prediction
else:
    criterion = nn.CrossEntropyLoss()
    # Assigns image to one class or the other based on raw scores
    
optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
# gains access to convolution filters, fully connected weights, and biases
# lr controls size of each weight update
# momentum allows SGD to move in directions with consistent reduced loss

### Training Model

In [23]:
for epoch in range(NUM_EPOCHS):
    train_correct = 0
    train_total = 0
    test_correct = 0
    test_total = 0
    
    model.train()
    for inputs, targets in tqdm(train_loader):
        # forward + backward + optimize
        optimizer.zero_grad() 
        # PyTorch adds gradients together. Clears leftover gradients for each batch

        outputs = model(inputs) # All output scores of batch
        
        if task == 'multi-label, binary-class':
            targets = targets.to(torch.float32)
            loss = criterion(outputs, targets)
        else:
            targets = targets.squeeze().long()
            loss = criterion(outputs, targets) # Calculates error
        
        loss.backward() # Runs backpropagation (computes which changes will reduce loss)
        optimizer.step() # Uses gradients to update model's weight.

100%|██████████| 37/37 [00:02<00:00, 12.68it/s]


### Evaluate Model

In [ ]:
def test(split):
    model.eval() # Puts model in evaluation mode
    y_true = torch.tensor([]) # Will hold all correct labels
    y_score = torch.tensor([]) # Will hold all model prediction scores
    
    data_loader = train_loader_at_eval if split == 'train' else test_loader

    with torch.no_grad(): # Tells PyTorch to not calculate gradients being we aren't training
        for inputs, targets in data_loader:
            outputs = model(inputs)

            if task == 'multi-label, binary-class':
                targets = targets.to(torch.float32)
                outputs = outputs.softmax(dim=-1)
            else:
                targets = targets.squeeze().long()
                outputs = outputs.softmax(dim=-1) # Converts raw scores to probabilities
                targets = targets.float().resize_(len(targets), 1)
                # Converts labels to column of decimals for evaluator

            y_true = torch.cat((y_true, targets), 0) 
            y_score = torch.cat((y_score, outputs), 0)
            # Appends each input, target pair (0 for rows)

        y_true = y_true.numpy()
        y_score = y_score.numpy()
        # Evaluator uses NumPy data, converts to NumPy arrays.
        
        evaluator = Evaluator(data_flag, split, root=str(DATA_ROOT))
        metrics = evaluator.evaluate(y_score)
    
        print('%s  auc: %.3f  acc:%.3f' % (split, *metrics))
        # AUC: How well the model ranks pneumonia images above normal images
        # ACC: Percent of correct predictions

        
print('==> Evaluating ...')
test('train')
test('test')

==> Evaluating ...
train  auc: 0.987  acc:0.947
test  auc: 0.944  acc:0.832
